# B1: Weight tour

Before writing any model code we look at exactly what is inside the checkpoint: every tensor's name and shape, where the 1.1B parameters sit, and what that means for memory.

Our own model (Part B) must load these tensors **by name**, so `tinyserve/weights.py` holds a pure-Python *spec* of every name and shape. Here we check the spec against the real model.

Questions this lesson answers:
1. What are the 201 tensors?
2. Which part of the model holds most of the bytes (and therefore most of each decode step's memory traffic, Lesson A2)?
3. What does grouped-query attention (GQA) change in the shapes, and what does the KV cache cost per token?

In [1]:
# Setup: clone or hard-reset the repo to GitHub, cd into it, put it on sys.path, install pytest.
import os, subprocess, sys

REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    # Colab may hold untracked files (e.g. results written by a notebook) that clash with a pull.
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

b33a6df B1: weight spec (names, shapes, param/KV math), tests, notebook



In [2]:
import importlib, json
import torch
import tinyserve.basics, tinyserve.quiet, tinyserve.weights
for module in (tinyserve.quiet, tinyserve.basics, tinyserve.weights):
    importlib.reload(module)  # picks up repo edits without restarting the kernel
from tinyserve.basics import load_model
from tinyserve.envinfo import collect_env
from tinyserve import weights

print(collect_env())
tokenizer, model = load_model(device="cuda", dtype=torch.float16)
config = model.config
print(f"layers {config.num_hidden_layers}, hidden {config.hidden_size}, MLP width {config.intermediate_size}, "
      f"query heads {config.num_attention_heads}, KV heads {config.num_key_value_heads}, head_dim {weights.head_dim(config)}, "
      f"vocab {config.vocab_size}, tied embeddings {config.tie_word_embeddings}")

{'python': '3.13.16', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'git_sha': 'b33a6df', 'gpu': 'Tesla T4', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'transformers': '5.20.2'}
layers 22, hidden 2048, MLP width 5632, query heads 32, KV heads 4, head_dim 64, vocab 32000, tied embeddings False


## 1. The tensors
A Llama checkpoint is a flat dictionary `name -> tensor`. There is one tensor for the token embedding, nine per decoder layer, a final norm, and an `lm_head`. With 22 layers that is `1 + 9*22 + 1 + 1 = 201`.

In [3]:
rows = weights.weight_tour(model.named_parameters())
print("tensors:", len(rows), "| dtypes:", {row["dtype"] for row in rows})
print()
for row in rows[:1] + rows[1:10] + rows[-2:]:
    print(f"{row['name']:<55} {str(row['shape']):>14} {row['numel']:>12,}")

tensors: 201 | dtypes: {'torch.float16'}

model.embed_tokens.weight                                (32000, 2048)   65,536,000
model.layers.0.self_attn.q_proj.weight                    (2048, 2048)    4,194,304
model.layers.0.self_attn.k_proj.weight                     (256, 2048)      524,288
model.layers.0.self_attn.v_proj.weight                     (256, 2048)      524,288
model.layers.0.self_attn.o_proj.weight                    (2048, 2048)    4,194,304
model.layers.0.mlp.gate_proj.weight                       (5632, 2048)   11,534,336
model.layers.0.mlp.up_proj.weight                         (5632, 2048)   11,534,336
model.layers.0.mlp.down_proj.weight                       (2048, 5632)   11,534,336
model.layers.0.input_layernorm.weight                          (2048,)        2,048
model.layers.0.post_attention_layernorm.weight                 (2048,)        2,048
model.norm.weight                                              (2048,)        2,048
lm_head.weight                    

## 2. Spec vs reality
`expected_shapes(config)` builds the same dictionary from the config alone. If any name is missing, unexpected, or has a different shape, `compare_shapes` tells us. B8 will rely on this spec.

In [4]:
actual_shapes = {row["name"]: row["shape"] for row in rows}
diff = weights.compare_shapes(actual_shapes, weights.expected_shapes(config))
print("differences:", diff)
assert diff == {"missing": [], "unexpected": [], "wrong_shape": []}

actual_total = sum(row["numel"] for row in rows)
print(f"parameters: actual {actual_total:,}  spec {weights.expected_param_count(config):,}")
assert actual_total == weights.expected_param_count(config)

layer0 = sum(row["numel"] for row in rows if row["name"].startswith("model.layers.0."))
print(f"one decoder layer: {layer0:,} parameters  x {config.num_hidden_layers} layers = {layer0 * config.num_hidden_layers:,}")

differences: {'missing': [], 'unexpected': [], 'wrong_shape': []}
parameters: actual 1,100,048,384  spec 1,100,048,384
one decoder layer: 44,044,288 parameters  x 22 layers = 968,974,336


## 3. Where the bytes are
In a decode step every weight is read once (Lesson A2), so the share of bytes is also the share of memory traffic.

In [5]:
totals = weights.bytes_by_category(rows)
for name, entry in sorted(totals.items(), key=lambda item: -item[1]["nbytes"]):
    print(f"{name:<10} {entry['numel'] / 1e6:9.1f} M params  {entry['nbytes'] / 1e6:9.1f} MB  {entry['share']:6.1%}")
print(f"{'total':<10} {sum(e['numel'] for e in totals.values()) / 1e6:9.1f} M params  {sum(e['nbytes'] for e in totals.values()) / 1e6:9.1f} MB")

mlp            761.3 M params     1522.5 MB   69.2%
attention      207.6 M params      415.2 MB   18.9%
embedding       65.5 M params      131.1 MB    6.0%
lm_head         65.5 M params      131.1 MB    6.0%
norm             0.1 M params        0.2 MB    0.0%
total         1100.0 M params     2200.1 MB


## 4. GQA and the KV cache
The 32 query heads share only 4 key/value heads (8 queries per KV head). So `k_proj` and `v_proj` output `4 x 64 = 256` features instead of 2048. That makes those projections 8x smaller, and, more importantly, the **KV cache 8x smaller**: for every token we store K and V for each layer.

`KV bytes per token = 2 (K and V) x layers x KV heads x head_dim x bytes per element`.

In [6]:
per_token = weights.kv_cache_bytes_per_token(config, bytes_per_element=2)
mha_per_token = 2 * config.num_hidden_layers * config.num_attention_heads * weights.head_dim(config) * 2
print(f"KV cache per token (FP16): GQA {per_token:,} bytes = {per_token / 1024:.0f} KiB; full multi-head would be {mha_per_token:,} bytes = {mha_per_token / 1024:.0f} KiB")
for context in (128, 2048):
    print(f"  one sequence of {context:>4} tokens: {per_token * context / 2**20:7.1f} MiB   (weights: {sum(r['nbytes'] for r in rows) / 2**20:.0f} MiB)")
print(f"sequences of 2048 tokens that fit in 10 GiB of free memory: {int(10 * 2**30 // (per_token * 2048))}")

KV cache per token (FP16): GQA 22,528 bytes = 22 KiB; full multi-head would be 180,224 bytes = 176 KiB
  one sequence of  128 tokens:     2.8 MiB   (weights: 2098 MiB)
  one sequence of 2048 tokens:    44.0 MiB   (weights: 2098 MiB)
sequences of 2048 tokens that fit in 10 GiB of free memory: 232


## 5. What the weights look like
Statistics of a few real tensors (computed in FP32). Norm weights are scales around 1 or smaller; projection weights are small numbers centred on 0.

In [7]:
params = dict(model.named_parameters())
stats_rows = {}
for name in ("model.embed_tokens.weight", "model.layers.0.self_attn.q_proj.weight", "model.layers.0.mlp.down_proj.weight",
             "model.layers.0.input_layernorm.weight", "model.layers.21.input_layernorm.weight", "model.norm.weight", "lm_head.weight"):
    stats = weights.tensor_stats(params[name])
    stats_rows[name] = stats
    print(f"{name:<45} mean {stats['mean']:8.4f}  std {stats['std']:7.4f}  min {stats['min']:8.3f}  max {stats['max']:8.3f}")

model.embed_tokens.weight                     mean  -0.0000  std  0.0149  min   -0.117  max    0.149
model.layers.0.self_attn.q_proj.weight        mean  -0.0000  std  0.0164  min   -1.586  max    1.031
model.layers.0.mlp.down_proj.weight           mean   0.0000  std  0.0166  min   -0.523  max    0.414
model.layers.0.input_layernorm.weight         mean   0.0058  std  0.0460  min   -0.582  max    0.770
model.layers.21.input_layernorm.weight        mean   0.4637  std  0.0852  min    0.114  max    1.227
model.norm.weight                             mean   1.9149  std  0.1365  min    0.508  max    3.188
lm_head.weight                                mean  -0.0004  std  0.0247  min   -0.471  max    0.395


## 6. Save results and run the tests

In [8]:
from tinyserve.envinfo import save_result

saved_path = save_result("B1_weights", {
    "num_tensors": len(rows),
    "num_params": actual_total,
    "params_per_layer": layer0,
    "by_category": totals,
    "kv_bytes_per_token_fp16": per_token,
    "kv_bytes_per_token_fp16_if_mha": mha_per_token,
    "tensor_stats": stats_rows,
    "layer0_shapes": {row["name"]: row["shape"] for row in rows if row["name"].startswith("model.layers.0.")},
})
print(saved_path)

/content/tinyserve/results/B1_weights.json


In [9]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b1_weights.py", "tests/test_quiet.py", "-v"], capture_output=True, text=True)
print(result.stdout[-3500:], result.stderr[-1000:])

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/tinyserve
configfile: pyproject.toml
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.5
collecting ... collected 13 items

tests/test_b1_weights.py::test_tinyllama_has_201_tensors PASSED          [  7%]
tests/test_b1_weights.py::test_tinyllama_param_count_matches_measured PASSED [ 15%]
tests/test_b1_weights.py::test_one_layer_has_44_million_parameters PASSED [ 23%]
tests/test_b1_weights.py::test_gqa_shapes PASSED                         [ 30%]
tests/test_b1_weights.py::test_kv_cache_is_22_kib_per_token_in_fp16 PASSED [ 38%]
tests/test_b1_weights.py::test_head_dim_falls_back_to_hidden_over_heads PASSED [ 46%]
tests/test_b1_weights.py::test_tied_embeddings_drop_lm_head PASSED       [ 53%]
tests/test_b1_weights.py::test_compare_shapes_reports_each_kind_of_diffe

In [10]:
# Print the full results file so it can be copied into the repo from the saved notebook.
print(open(saved_path).read())

{
  "name": "B1_weights",
  "timestamp_utc": "2026-10-25T07:09:56Z",
  "env": {
    "python": "3.13.16",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "git_sha": "b33a6df",
    "gpu": "Tesla T4",
    "torch": "2.11.0+cu130",
    "cuda": "13.0",
    "transformers": "5.20.2"
  },
  "data": {
    "num_tensors": 201,
    "num_params": 1100048384,
    "params_per_layer": 44044288,
    "by_category": {
      "embedding": {
        "numel": 65536000,
        "nbytes": 131072000,
        "share": 0.0595755613600356
      },
      "attention": {
        "numel": 207618048,
        "nbytes": 415236096,
        "share": 0.18873537838859278
      },
      "mlp": {
        "numel": 761266176,
        "nbytes": 1522532352,
        "share": 0.6920297207581735
      },
      "norm": {
        "numel": 92160,
        "nbytes": 184320,
        "share": 8.377813316255006e-05
      },
      "lm_head": {
        "numel": 65536000,
        "nbytes": 131072000,
        "share": 0.05957556136003

In [ ]:
# Free the GPU for the next notebook: every kernel on this Colab server keeps its own copy of the model.
import gc
import sys
import torch
for name in ("model", "params"):
    globals().pop(name, None)
for name in ("last_traceback", "last_value", "last_type", "last_exc"):  # an old error can keep tensors alive
    if hasattr(sys, name):
        setattr(sys, name, None)
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")